# Lesson 01 - Exploring LangChain Chat Models

LangChain chat models let Python programs talk to AI models using structured messages. This folder starts with one simple model call, then builds toward conversation history, terminal chat loops, and Firebase-backed memory.

In this lesson, we will learn four core ideas:

- **Model** - the AI chat model object created with `ChatOpenAI`
- **Messages** - structured conversation items like `SystemMessage`, `HumanMessage`, and `AIMessage`
- **Session memory** - a Python list that remembers messages while the program is running
- **Persistent memory** - Firestore storage that remembers messages even after the program stops


## Setup

Before running these examples, make sure your virtual environment is active and the required packages are installed. The `.env` file should contain your OpenAI API key.


In [ ]:
# Install the main packages used in this folder
!pip install langchain-openai python-dotenv langchain-google-firestore google-cloud-firestore

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

load_dotenv()

llm = ChatOpenAI(model="gpt-5-nano")

## Understanding The Chat Model Flow

The examples in this folder follow a simple pattern:

```text
.env file -> load_dotenv() -> ChatOpenAI -> messages or prompt -> llm.invoke() -> response.content
```

1. `load_dotenv()` loads API keys from `.env`.
2. `ChatOpenAI(...)` creates the model object.
3. A prompt or message list is passed to `llm.invoke(...)`.
4. The model returns a response object.
5. `.content` extracts the readable answer text.

For beginners, the most important idea is that `llm.invoke(...)` is the line that actually asks the AI model for an answer.


## Example 1 - Basic Chat Model Call

File: `1_chat_models_starter.py`

This file shows the smallest useful LangChain chat model program. It creates a model, sends one prompt, and prints the answer.


In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(model="gpt-5-nano")

result = llm.invoke("Hello, how are you?")
print(result.content)

### What The Important Lines Do

- `load_dotenv()` loads environment variables from `.env`, especially `OPENAI_API_KEY`.
- `ChatOpenAI(model="gpt-5-nano")` creates the chat model object.
- `llm.invoke(...)` sends a prompt to the model.
- `result.content` contains the text answer.

This is the foundation for every later file in the folder.


## Example 2 - Conversation With Message Objects

Files: `2_chat_models_conversation.py` and `3_chat_models-alternative_models.py`

These files introduce message roles. Instead of sending one plain string, the code sends a list of structured messages.


In [ ]:
messages = [
    SystemMessage(content="You are a helpful assistant that can answer questions and help with tasks."),
    HumanMessage(content="What is the capital of France?"),
    AIMessage(content="Paris."),
    HumanMessage(content="What is the capital of Germany?"),
]

res = llm.invoke(messages)
print(res.content)

### Understanding Message Roles

- `SystemMessage` gives instructions to the assistant. It controls behavior.
- `HumanMessage` represents what the user said.
- `AIMessage` represents what the assistant already said.
- `messages = [...]` keeps the conversation in order.
- `llm.invoke(messages)` sends the full conversation context to the model.

The model can use earlier messages to answer the newest question. This is the basic idea behind chatbot memory.


## Example 3 - Interactive Terminal Chatbot

File: `4_chat_model_conversation_with_user.py`

This file turns the message-list idea into a chatbot that runs in the terminal.


In [ ]:
messages = [
    SystemMessage(content="You are a helpful assistant that can answer questions and help with tasks."),
]

while True:
    user_input = input("You: ")

    if user_input.lower() in ["quit", "exit", "bye"]:
        break

    messages.append(HumanMessage(content=user_input))
    response = llm.invoke(messages)
    messages.append(AIMessage(content=response.content))

    print(f"Assistant: {response.content}")

### How The Chat Loop Works

- `while True:` keeps the program running until we manually stop it.
- `input("You: ")` waits for the user to type a message.
- The `if` statement checks for exit words like `quit`, `exit`, or `bye`.
- `messages.append(HumanMessage(...))` saves the user's message.
- `llm.invoke(messages)` sends the full conversation to the model.
- `messages.append(AIMessage(...))` saves the assistant's reply for the next turn.

This memory is temporary. When the script stops, the `messages` list disappears.


## Example 4 - Saving Message History With Firebase

File: `5_chat_model_save_message_history_firebase.py`

This file stores chat history in Google Firestore so the conversation can continue across separate program runs.


In [ ]:
from google.cloud import firestore
from langchain_google_firestore import FirestoreChatMessageHistory

PROJECT_ID = "langchain-memory-4adbd"
SESSION_ID = "user_session_new"
COLLECTION_NAME = "chat_history"

client = firestore.Client(project=PROJECT_ID)

chat_history = FirestoreChatMessageHistory(
    session_id=SESSION_ID,
    collection=COLLECTION_NAME,
    client=client,
)

print("Current Chat History:", chat_history.messages)

### Important Firebase Lines

- `firestore.Client(project=PROJECT_ID)` connects Python to your Firebase/Google Cloud project.
- `SESSION_ID` identifies one conversation. Different session IDs can store different chats.
- `COLLECTION_NAME` is the Firestore collection where messages are saved.
- `FirestoreChatMessageHistory(...)` creates a LangChain memory object backed by Firestore.
- `chat_history.messages` loads the saved conversation history.
- `chat_history.add_user_message(...)` saves a user message.
- `chat_history.add_ai_message(...)` saves an assistant response.

The key difference is persistence. A Python list forgets messages when the program exits. Firestore keeps them.


## Folder Recap

By the end of this folder, beginners should understand this progression:

1. A chat model can answer one prompt.
2. A list of messages gives the model conversation context.
3. A loop turns the model into a terminal chatbot.
4. A Python list gives temporary memory.
5. Firestore gives persistent memory.

The most important functions and classes are `ChatOpenAI`, `load_dotenv()`, `llm.invoke(...)`, `SystemMessage`, `HumanMessage`, `AIMessage`, and `FirestoreChatMessageHistory`.
